In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/inventory_sales_data.csv', parse_dates=['Date'])

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

Rows: 36550, Columns: 15


,Product,Category,Warehouse,Supplier,Date,Season,Promotion,Price,Lead_Time,Raw_Demand,Historical_Sales,Returns,Current_Inventory,Lost_Sales,Stockout_Flag
0,P001,Electronics,WH_North,Supplier_C,2023-01-01,Winter,0,12714.12,5,11,11,0,99,0,0
1,P001,Electronics,WH_North,Supplier_C,2023-01-02,Winter,0,12714.12,5,15,15,0,84,0,0
2,P001,Electronics,WH_North,Supplier_C,2023-01-03,Winter,1,12714.12,5,24,24,0,60,0,0
3,P001,Electronics,WH_North,Supplier_C,2023-01-04,Winter,0,12714.12,5,17,17,0,43,0,0
4,P001,Electronics,WH_North,Supplier_C,2023-01-05,Winter,1,12714.12,5,20,20,0,23,0,0


In [2]:


print("=== Duplicate rows ===")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate Product+Date combos: {df.duplicated(subset=['Product','Date']).sum()}")

print("\n=== Missing values ===")
print(df.isnull().sum().sum(), "total missing values")

print("\n=== Negative value checks (should all be 0) ===")
print(f"Negative Historical_Sales: {(df['Historical_Sales'] < 0).sum()}")
print(f"Negative Current_Inventory: {(df['Current_Inventory'] < 0).sum()}")
print(f"Negative Price: {(df['Price'] < 0).sum()}")

print("\n=== Logical consistency ===")
print(f"Rows where Historical_Sales > Raw_Demand: {(df['Historical_Sales'] > df['Raw_Demand']).sum()}")
print(f"Rows where Lost_Sales != Raw_Demand - Historical_Sales: {(df['Lost_Sales'] != (df['Raw_Demand'] - df['Historical_Sales'])).sum()}")

print("\n=== Date coverage per product ===")
date_counts = df.groupby('Product')['Date'].count()
print(f"Expected days per product: 731")
print(f"Products with wrong day count: {(date_counts != 731).sum()}")

=== Duplicate rows ===
Exact duplicate rows: 0
Duplicate Product+Date combos: 0

=== Missing values ===
0 total missing values

=== Negative value checks (should all be 0) ===
Negative Historical_Sales: 0
Negative Current_Inventory: 0
Negative Price: 0

=== Logical consistency ===
Rows where Historical_Sales > Raw_Demand: 0
Rows where Lost_Sales != Raw_Demand - Historical_Sales: 0

=== Date coverage per product ===
Expected days per product: 731
Products with wrong day count: 0


In [3]:

df = df.sort_values(['Product', 'Date']).reset_index(drop=True)


df['Rolling_7d_Demand'] = df.groupby('Product')['Raw_Demand'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

df['Rolling_30d_Demand'] = df.groupby('Product')['Raw_Demand'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)

df[df['Product']=='P001'][['Date','Raw_Demand','Rolling_7d_Demand','Rolling_30d_Demand']].head(15)

,Date,Raw_Demand,Rolling_7d_Demand,Rolling_30d_Demand
0,2023-01-01,11,11.000000,11.000000
1,2023-01-02,15,13.000000,13.000000
2,2023-01-03,24,16.666667,16.666667
3,2023-01-04,17,16.750000,16.750000
4,2023-01-05,20,17.400000,17.400000
5,2023-01-06,12,16.500000,16.500000
6,2023-01-07,6,15.000000,15.000000
7,2023-01-08,17,15.857143,15.250000
8,2023-01-09,13,15.571429,15.000000
9,2023-01-10,12,13.857143,14.700000


In [4]:

df['Days_Stock_Remaining'] = np.where(
    df['Rolling_7d_Demand'] > 0,
    df['Current_Inventory'] / df['Rolling_7d_Demand'],
    999  # placeholder for "no recent demand, not at risk"
)

df[df['Product']=='P001'][['Date','Current_Inventory','Rolling_7d_Demand','Days_Stock_Remaining']].head(15)

,Date,Current_Inventory,Rolling_7d_Demand,Days_Stock_Remaining
0,2023-01-01,99,11.000000,9.000000
1,2023-01-02,84,13.000000,6.461538
2,2023-01-03,60,16.666667,3.600000
3,2023-01-04,43,16.750000,2.567164
4,2023-01-05,23,17.400000,1.321839
5,2023-01-06,11,16.500000,0.666667
6,2023-01-07,5,15.000000,0.333333
7,2023-01-08,0,15.857143,0.000000
8,2023-01-09,0,15.571429,0.000000
9,2023-01-10,0,13.857143,0.000000


In [5]:

df['Expected_Demand_During_LeadTime'] = df['Rolling_7d_Demand'] * df['Lead_Time']


df['At_Risk_Flag'] = (df['Current_Inventory'] < df['Expected_Demand_During_LeadTime']).astype(int)

df['Inventory_Gap'] = df['Current_Inventory'] - df['Expected_Demand_During_LeadTime']

df[df['Product']=='P001'][['Date','Current_Inventory','Lead_Time','Expected_Demand_During_LeadTime','At_Risk_Flag','Inventory_Gap']].head(15)

,Date,Current_Inventory,Lead_Time,Expected_Demand_During_LeadTime,At_Risk_Flag,Inventory_Gap
0,2023-01-01,99,5,55.000000,0,44.000000
1,2023-01-02,84,5,65.000000,0,19.000000
2,2023-01-03,60,5,83.333333,1,-23.333333
3,2023-01-04,43,5,83.750000,1,-40.750000
4,2023-01-05,23,5,87.000000,1,-64.000000
5,2023-01-06,11,5,82.500000,1,-71.500000
6,2023-01-07,5,5,75.000000,1,-70.000000
7,2023-01-08,0,5,79.285714,1,-79.285714
8,2023-01-09,0,5,77.857143,1,-77.857143
9,2023-01-10,0,5,69.285714,1,-69.285714


In [6]:

output_path = '../data/inventory_data_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows, {len(df.columns)} columns to {output_path}")
print(f"\nNew columns added today: Rolling_7d_Demand, Rolling_30d_Demand, Days_Stock_Remaining,")
print(f"Expected_Demand_During_LeadTime, At_Risk_Flag, Inventory_Gap")

print(f"\nOverall At_Risk_Flag rate: {df['At_Risk_Flag'].mean()*100:.1f}% of product-days currently flagged at-risk")

Saved 36550 rows, 21 columns to ../data/inventory_data_cleaned.csv

New columns added today: Rolling_7d_Demand, Rolling_30d_Demand, Days_Stock_Remaining,
Expected_Demand_During_LeadTime, At_Risk_Flag, Inventory_Gap

Overall At_Risk_Flag rate: 79.3% of product-days currently flagged at-risk
